# Temporal graph inspection

Build and inspect the hand-checkable CPU fixture rather than loading the full ICEWS14 source. The fixture has two directed two-hop paths, distinct event dates, and a missing seed entity. The cutoff traversal below is a notebook-only inspection; production temporal filtering is a later task.

In [ ]:
import json
from datetime import date
from pathlib import Path

from src.graph.graph_builder import build_temporal_graph

In [ ]:
fixture_relative_path = Path("tests") / "fixtures" / "temporal_fixture.json"
fixture_path = next(
    (
        base / fixture_relative_path
        for base in (Path.cwd(), *Path.cwd().parents)
        if (base / fixture_relative_path).is_file()
    ),
    None,
)
if fixture_path is None:
    raise FileNotFoundError(f"Could not locate {fixture_relative_path} from {Path.cwd()}")

fixture = json.loads(fixture_path.read_text(encoding="utf-8"))
graph = build_temporal_graph(fixture["facts"])
print(f"Fixture: {fixture['fixture_id']}")
print(f"Entities: {len(graph.entity_ids)} | relations: {len(graph.relation_ids)} | facts: {graph.fact_count}")

In [ ]:
print("Dated directed facts")
for fact in graph.facts:
    print(
        f"{fact['timestamp']}  {fact['subject_id']} "
        f"-[{fact['relation_id']}]-> {fact['object_id']} "
        f"(fact_id={fact['fact_id']})"
    )

In [ ]:
seed = "entity_a"
print(f"Outgoing from {seed}:")
for fact in graph.outgoing_facts(seed):
    print(f"  {fact['fact_id']}: {fact['relation_id']} -> {fact['object_id']} ({fact['timestamp']})")

print("\nIncoming to entity_c:")
for fact in graph.incoming_facts("entity_c"):
    print(f"  {fact['fact_id']}: {fact['subject_id']} -> entity_c ({fact['timestamp']})")

Trace each fixture query using the graph's outgoing-fact index. Dates are compared inclusively here to make the fixture's boundary behavior visible; this small traversal is not a replacement for the planned retrieval module.

In [ ]:
def trace_paths(seed_entity_id, relation_sequence, cutoff_date):
    cutoff = date.fromisoformat(cutoff_date)
    paths = [(seed_entity_id, ())]
    for relation_id in relation_sequence:
        next_paths = []
        for entity_id, fact_ids in paths:
            for fact in graph.outgoing_facts(entity_id, relation_id):
                if date.fromisoformat(fact["timestamp"]) <= cutoff:
                    next_paths.append((fact["object_id"], fact_ids + (fact["fact_id"],)))
        paths = next_paths
    return paths


for query in fixture["queries"]:
    paths = trace_paths(
        query["seed_entity_id"], query["relation_sequence"], query["cutoff_date"]
    )
    path_ids = sorted([list(fact_ids) for _, fact_ids in paths])
    answer_ids = sorted({entity_id for entity_id, _ in paths})
    assert path_ids == sorted(query["expected_paths"]), query["query_id"]
    assert answer_ids == sorted(query["expected_answer_ids"]), query["query_id"]
    print(f"{query['query_id']} (cutoff {query['cutoff_date']}):")
    print(f"  paths={path_ids} answers={answer_ids}")